In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import earthaccess
import tempfile

NOTE: I have had to do multiple instillations to get this going. I added to the conda environment:
- geopandas
- earthaccess
- pyarrow
- jupyter notebook

In [2]:
COLUMNS = [
    "YEAR", "DOY", "SOD",
    "LON", "LAT",
    "ICE_THICK", "SRF_RNG", "BED_ELEVATION", "SURFACE_ELEVATION",
    "BED_REFLECT", "SRF_REFLECT", "AIRCRAFT_ROLL"
]

In [3]:
earthaccess.login()

Enter your Earthdata Login username:  roma8902
Enter your Earthdata password:  ········


In [4]:
def build_geoparquet(input_dir: str | Path, out_path: str | Path):
    """
    Batch-convert a directory of IceBridge HiCARS ice thickness text files
    to a single GeoParquet file.

    Reads all ``*.txt`` files in ``input_dir``, concatenates them, drops
    rows with missing coordinates, and constructs point geometries from
    LON/LAT (EPSG:4326). The data is reprojected to Antarctic polar
    stereographic (EPSG:3031) and projected ``X``/``Y`` coordinates are
    written as explicit columns alongside the geometry for direct use
    with gstatsim.

    Parameters
    ----------
    input_dir : str or Path
        Directory containing ``*_icethk.txt`` files (IR1HI2 and/or IR2HI2).
    out_path : str or Path
        Destination path for the output GeoParquet file.

    Returns
    -------
    gpd.GeoDataFrame
        GeoDataFrame in EPSG:3031 with all input columns plus ``X``, ``Y``,
        ``source_file``, and ``instrument``.
    """
    input_dir = Path(input_dir)
    files = sorted(input_dir.glob("*.txt"))
    print(f"Found {len(files)} files")

    chunks = [read_icethk(f) for f in files]
    df = pd.concat(chunks, ignore_index=True)
    print(f"Total rows (including NaN positions): {len(df):,}")

    df = df.dropna(subset=["LON", "LAT"])
    print(f"Rows with valid positions: {len(df):,}")

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["LON"], df["LAT"]),
        crs="EPSG:4326",
    )

    gdf = gdf.to_crs("EPSG:3031")
    gdf["X"] = gdf.geometry.x
    gdf["Y"] = gdf.geometry.y

    gdf.to_parquet(out_path, compression="snappy")
    print(f"Written → {out_path}")
    return gdf

In [5]:
def read_icethk(path: Path) -> pd.DataFrame:
    """
    Read a single IceBridge HiCARS ice thickness text file into a DataFrame.

    Parses the variable-length comment header by scanning for the
    ``Length_of_header`` metadata field, then reads the space-delimited
    data block. Missing values (``nan`` in the raw files) are preserved
    as ``NaN``. A ``source_file`` column and an ``instrument`` column
    (``HiCARS1`` or ``HiCARS2``, inferred from the filename prefix) are
    appended for provenance tracking.

    Parameters
    ----------
    path : Path
        Path to a ``*_icethk.txt`` file (IR1HI2 or IR2HI2 format).

    Returns
    -------
    pd.DataFrame
        DataFrame with columns: YEAR, DOY, SOD, LON, LAT, ICE_THICK,
        SRF_RNG, BED_ELEVATION, SURFACE_ELEVATION, BED_REFLECT,
        SRF_REFLECT, AIRCRAFT_ROLL, source_file, instrument.
        Rows with missing LON/LAT are retained at this stage.
    """
    with open(path) as f:
        for i, line in enumerate(f):
            if "Length_of_header:" in line:
                header_lines = int(line.split(":")[-1].strip().split()[0])
                break

    df = pd.read_csv(
        path,
        skiprows=header_lines,
        sep=r"\s+",
        names=COLUMNS,
        na_values="nan",
    )
    df["source_file"] = path.name
    df["instrument"] = "HiCARS1" if "IR1HI2" in path.name else "HiCARS2"
    return df

In [10]:
out_path = Path("data/icethk_all.parquet").expanduser()
out_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# Dataset landing pages:
# https://nsidc.org/data/ir1hi2/versions/1
# https://nsidc.org/data/ir2hi2/versions/1


with tempfile.TemporaryDirectory() as tmpdir:
    for concept_id in ["C3204979277-NSIDC_CPRD", "C3204982997-NSIDC_CPRD"]:
        results = earthaccess.search_data(concept_id=concept_id)
        print(f"{concept_id}: {len(results)} granules")
        earthaccess.download(results, local_path=tmpdir)
        
    gdf = build_geoparquet(tmpdir, out_path)

C3204979277-NSIDC_CPRD: 243 granules


QUEUEING TASKS | :   0%|          | 0/243 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/243 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/243 [00:00<?, ?it/s]

C3204982997-NSIDC_CPRD: 617 granules


QUEUEING TASKS | :   0%|          | 0/617 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/617 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/617 [00:00<?, ?it/s]

Found 860 files


In [ ]:
gdf = gpd.read_parquet("data/icethk_all.parquet")
print(gdf.shape)
print(gdf.dtypes)
gdf.head()